# DP-FedLoRA GPU Experiments

Runs the three P0 experiments needed to fill `\needsevidence{}` blocks in the paper.

**Runtime estimate on T4 (Colab/Kaggle):** ~3–5 hours for all three runs at reduced scale (5 clients, 10 rounds, full data). Set `FULL_SCALE=True` for the paper's 10-client / 20-round config (~8–12 hours).

After all cells complete, **copy the final JSON block** and paste it back to Claude.

In [ ]:
# ---- 0. GPU check ----
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NONE — abort and switch to GPU runtime')

import torch
print('torch CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA — switch runtime to GPU before continuing'

In [ ]:
# ---- 1. Install uv ----
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']
!uv --version

In [ ]:
# ---- 2. Clone repo ----
!git clone https://github.com/arulmr0/dp-fedlora.git
%cd dp-fedlora
!git log --oneline -3

In [ ]:
# ---- 3. Install dependencies ----
!uv sync
# Verify key packages
!uv run python -c "import torch, opacus, peft, transformers; print('torch', torch.__version__, '| opacus', opacus.__version__, '| peft', peft.__version__)"

In [ ]:
# ---- 4. Scale config ----
# FULL_SCALE=True  → paper config (10 clients, 20 rounds) ~8-12h on T4
# FULL_SCALE=False → reduced config (5 clients, 10 rounds) ~3-5h on T4
FULL_SCALE = False

if FULL_SCALE:
    CLIENTS = 10
    ROUNDS  = 20
    SUFFIX  = 'fullscale'
else:
    CLIENTS = 5
    ROUNDS  = 10
    SUFFIX  = 'gpu5c10r'

print(f'Running: {CLIENTS} clients, {ROUNDS} rounds  [FULL_SCALE={FULL_SCALE}]')

In [ ]:
# ---- 5a. Experiment 1: FedAvg no-DP (upper bound) ----
EXP1 = f'gpu_fedavg_nodp_{SUFFIX}'
print(f'=== Experiment 1: FedAvg no-DP  [{EXP1}] ===')
!uv run python pipeline/train.py \
    dataset.num_clients={CLIENTS} \
    fl.num_rounds={ROUNDS} \
    experiment.name={EXP1}

In [ ]:
# ---- 5b. Experiment 2: FedAvg + DP-SGD (full-model DP baseline) ----
EXP2 = f'gpu_fedavg_dp_{SUFFIX}'
print(f'=== Experiment 2: FedAvg + DP-SGD  [{EXP2}] ===')
!uv run python pipeline/train.py \
    dataset.num_clients={CLIENTS} \
    fl.num_rounds={ROUNDS} \
    privacy=dp_sgd \
    experiment.name={EXP2}

In [ ]:
# ---- 5c. Experiment 3: DP-FedLoRA (adapter-only DP) ----
EXP3 = f'gpu_fedlora_dp_{SUFFIX}'
print(f'=== Experiment 3: DP-FedLoRA r=16  [{EXP3}] ===')
!uv run python pipeline/train.py \
    dataset.num_clients={CLIENTS} \
    fl.num_rounds={ROUNDS} \
    model=vit_b16_lora \
    privacy=dp_sgd \
    experiment.name={EXP3}

In [ ]:
# ---- 6. Print results for copy-paste ----
import json, pathlib

exps = [EXP1, EXP2, EXP3]
summary = {}

for name in exps:
    p = pathlib.Path(f'outputs/{name}/results.json')
    if p.exists():
        d = json.loads(p.read_text())
        tm = d['test_metrics']
        rounds = d['history']['rounds']
        eps_vals = [r['epsilon'] for r in rounds if r.get('epsilon') is not None]
        summary[name] = {
            'test_auc':   round(tm['auc'], 4),
            'test_acc':   round(tm['accuracy'], 4),
            'final_eps':  round(eps_vals[-1], 3) if eps_vals else None,
            'val_auc_history': [round(r['val_auc'], 4) for r in rounds],
            'seconds_per_round': [round(r['seconds'], 1) for r in rounds],
            'num_rounds': len(rounds),
            'num_clients': CLIENTS,
            'full_scale': FULL_SCALE,
        }
    else:
        summary[name] = {'error': 'results.json not found'}

print('\n' + '='*60)
print('PASTE THIS BLOCK BACK TO CLAUDE')
print('='*60)
print(json.dumps(summary, indent=2))
print('='*60)

In [ ]:
# ---- 7. (Optional) Download results.json files ----
# On Colab: this downloads to your browser
try:
    from google.colab import files
    import shutil, pathlib
    for name in exps:
        p = pathlib.Path(f'outputs/{name}/results.json')
        if p.exists():
            dest = f'{name}_results.json'
            shutil.copy(p, dest)
            files.download(dest)
except ImportError:
    # Kaggle: results are in /kaggle/working/dp-fedlora/outputs/
    print('Not on Colab. On Kaggle, find results in the Output tab.')
    for name in exps:
        p = pathlib.Path(f'outputs/{name}/results.json')
        print(f'  {p}:', 'EXISTS' if p.exists() else 'MISSING')